In [87]:
import pandas as pd
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)    # Show all rows
pd.set_option('display.max_columns', None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [88]:
#get price data
price = pd.read_parquet("data/spy.parquet")
price_close = price['Close'].dropna()
#print(price_close.head(30))

# get first actual trading day of each month
monthly = price_close.groupby(pd.Grouper(freq="MS")).nth(0)
# groupby() similar to SQL GROUP BY with rules/columns
# Grouper() = build in function to define grouping rules, here we use "MS" for monthly interval (month start)

#monthly = price_close.groupby(pd.Grouper(freq="MS")).first()
#nth() apply to df index, then get the first data, no matter NA or not
#first() apply to group index, then get the first available data from df

#resample() is similar to groupby() + Grouper(), but resample() only works on time series data,
#and it will automatically fill missing values with NaN, while groupby() + Grouper() will not fill missing values,
# and it will only return the existing data in the original dataframe.
"""
rule = pd.Grouper(freq="MS")     # define grouping rule
grouped = price_close.groupby(rule)  # apply grouping
monthly = grouped.nth(0)         # take first item in each group
"""
# 對齊回每日 date index ~ to left join / vlookup, index = main table, reindex = second table
price_close["monthly_price"] = monthly.reindex(price_close.index)

In [89]:
investment = 1000 # 每月投入金額
shares = investment / monthly  # 每次買幾多股, python will boardcast the division across the series/df, resulting in a series of shares bought
cum_shares = shares.cumsum() # 累積持有的股數

# 算出每日股數, 先對齊回每日 index, 再用 forward fill 填補缺失值 (因為月初買入後, 每日持有的股數不變)
price_close['daily_cum_shares'] = cum_shares.reindex(price_close.index, method="ffill") 
# 計算出每日的portfolio value
price_close['port_value'] = price_close['daily_cum_shares'] * price_close['SPY']

In [90]:
#IRR 計算
total_invested = investment * len(monthly) #total invested amount, positive for calculation
cashflows = [-investment] * len(monthly) #total cash outflow with series, negative for investment 
cashflows[-1] += price_close['port_value'].iloc[-1] #假設最後一日賣出所有股票並加到cashflow中, IRR 計法
#iloc is used to locate the row by its integer position, -1 = the last row

monthly_irr = npf.irr(cashflows) #np financial function 
annualized_irr = (1 + monthly_irr) ** 12 - 1 #Annualized is important**

#CAGR 計算 (for lum sum usually)
years = (price.index[-1] - price.index[0]).days / 365 #find the total years
cagr = (price_close['port_value'].iloc[-1] / total_invested) ** (1 / years) - 1 
#CAGR = (final value / initial value) ^ (1 / number of years) - 1

In [ ]:
# 其他指標計算 - cashflow series for actual calculation
# 建立 cashflow series（只有月初有）
cashflow_series = pd.Series(0, index=price_close.index) #建立全0的Series, index與price_close相同
cashflow_series.loc[monthly.index] = investment #locate monthly date (index), assign investment amount to those dates, 其他日期仍為0

# return 
# returns = (current value - previous value) / previous value but include cashflow impact for strategy return
# .pct_change() is a built-in function to calculate percentage change between the current and a prior element
# price_close['returns'] = price_close['port_value'].pct_change() *100

# daily real return = (current value - previous value - cashflow) / previous value
# shift = move the data down by n row
price_close['strategy_return'] = (
    price_close['port_value'] - price_close['port_value'].shift(1) - cashflow_series
) / price_close['port_value'].shift(1) 

# ===== SHARPE =====
ret = price_close['strategy_return'].dropna()

mean_return = ret.mean()
std_return = ret.std()

sharpe = mean_return / std_return * (252 ** 0.5)

# ===== SORTINO =====
downside = price_close['strategy_return'][price_close['strategy_return'] < 0]
# 1st[] original return
# 2nd [] filter return is negative, then it's a downside
# combine into downside varible
sortino = mean_return / downside.std() * (252 ** 0.5)

# ===== VOLATILITY =====
volatility = std_return * (252 ** 0.5)

# ===== MDD =====
price_close['cum_max'] = price_close['port_value'].cummax()
price_close['drawdown'] = price_close['port_value'] / price_close['cum_max'] - 1
mdd = price_close['drawdown'].min()

In [92]:
print(price_close['port_value'].iloc[-1].round(2))
print(f"Total invested: {total_invested}")
print(f"Annualized IRR: {annualized_irr:.2%}")
#print(f"CAGR: {cagr:.2%}")
print(f"MDD: {mdd:.2%}")
print(f"Sharpe: {sharpe:.2f}")
print(f"Sortino: {sortino:.2f}")
print(f"Volatility: {volatility:.2%}")

1067441.08
Total invested: 240000
Annualized IRR: 13.31%
MDD: -36.17%
Sharpe: 0.63
Sortino: 0.76
Volatility: 1938.61%
